# ⚡ TerraZ: Enterprise AI for Ethical EV Mineral Exploration
### **IBM Z Datathon 2026 | Theme: AI for Good**
---
**TerraZ** runs natively on **IBM LinuxONE / IBM z16 (s390x architecture)**, pairing deep learning prospectivity with real-time **ESG risk modeling, water stress analytics, carbon footprints, and Quantum-Safe cryptographic mineral passports**.

In [ ]:
import sys
import os
sys.path.insert(0, '/home/ibm-user/shared/EV_mineral_exploration')

import torch
import joblib
import pandas as pd
import numpy as np

from esg_engine import calculate_esg_scorecard, get_global_esg_matrix, COUNTRY_ESG_DATA, COMMODITY_CARBON_PROFILES
from ibm_z_engine import calculate_linuxone_sustainability, generate_quantum_safe_mineral_passport, verify_quantum_safe_mineral_passport, simulate_tamper_attempt, LATENCY_BENCHMARKS
from xai_engine import explain_prediction

print("TerraZ Core Engines Loaded Successfully on IBM LinuxONE s390x!")
print(f"PyTorch: {torch.__version__} | OS: Linux s390x")

## 1. Load 100,000-Row Global Critical Mineral Dataset

In [ ]:
df = pd.read_csv('/home/ibm-user/shared/EV_mineral_exploration/dataset.csv', keep_default_na=False)
print(f"Total Verified Occurrences: {len(df):,}")
print(f"Global Nations Represented: {df['country'].nunique()}")
print(f"EV Minerals: {df['commod'].nunique()}")
df.head()

## 2. Real-Time Deep Learning Inference & Plain-Language XAI

In [ ]:
import torch.nn as nn

class MineralNet(nn.Module):
    def __init__(self, input_dim=3, hidden_dim=64, output_dim=12):
        super(MineralNet, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.dropout = nn.Dropout(0.2)
        self.fc3 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

le = joblib.load('/home/ibm-user/shared/EV_mineral_exploration/models/label_encoder.pkl')
scaler = joblib.load('/home/ibm-user/shared/EV_mineral_exploration/models/scaler.pkl')
model = MineralNet(input_dim=3, hidden_dim=64, output_dim=len(le.classes_))
model.load_state_dict(torch.load('/home/ibm-user/shared/EV_mineral_exploration/models/mineral_dl_model.pth', map_location='cpu'))
model.eval()

# Test Prediction on Chile Centroid
lat, lon, grade_val = -23.5, -68.3, 1
feat = scaler.transform([[lat, lon, grade_val]])
with torch.no_grad():
    probs = torch.softmax(model(torch.tensor(feat, dtype=torch.float32)), dim=1).numpy().flatten()

pred_mineral = le.inverse_transform([np.argmax(probs)])[0]
xai = explain_prediction(model, scaler, le, lat, lon, grade_val, 'Chile', pred_mineral)

print(f"AI Prediction: {pred_mineral} ({max(probs)*100:.1f}% Confidence)")
print(f"Plain-English Explanation: {xai['plain_summary']}")

## 3. ESG & Ethical Discovery Scorecard (UN SDGs 7, 12, 13, 15)

In [ ]:
scorecard = calculate_esg_scorecard('Chile', pred_mineral, 'A')
for k, v in scorecard.items():
    print(f"{k}: {v}")

## 4. IBM LinuxONE Green Compute & Quantum-Safe Provenance

In [ ]:
# LinuxONE Sustainability Savings (50 servers consolidated)
savings = calculate_linuxone_sustainability(50)
print(f"Annual Power Reduction: {savings['kwh_saved_annually']:,} kWh (-{savings['reduction_percentage']}%) ")
print(f"5-Year Energy Cost Savings: ${savings['five_year_savings_usd']:,.2f}")
print(f"Equivalent Passenger Cars Removed: {savings['cars_removed_equivalent']} cars/year")

# Mint Quantum-Safe Mineral Passport
passport = generate_quantum_safe_mineral_passport('Salar de Atacama Block 1', pred_mineral, 'Chile', 'A', scorecard)
print(f"\nMinted Passport: {passport['passport_id']}")
print(f"Lattice Signature: {passport['quantum_signature']}")

# Verify Authenticity with IBM Z HSM Key
audit = verify_quantum_safe_mineral_passport(passport)
print(f"Cryptographic Audit Result: {audit['security_status']}")

# Test Anti-Tampering Shield
tamper_test = simulate_tamper_attempt(passport, 'eedi_score', 99.9)
print(f"Tamper-Detection Shield: {'SUCCESSFULLY BLOCKED' if tamper_test['detected_by_security'] else 'FAILED'}")